### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 — OpenTelemetry 기반 Span Evaluation
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar)**

> 목표: 에이전트 실행을 **OpenTelemetry Trace/Span**으로 계측하고, **Span attribute**(모델·토큰·도구·지연)를 수집해
> **Span 단위로 평가**합니다. 프레임워크에 독립적인(framework-agnostic) 평가의 토대입니다.

## 0. 셋업
Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`를 등록한 뒤, 아래 셀을 실행합니다.

In [1]:
!pip -q install openai opentelemetry-sdk

In [2]:
# Upstage Solar 셋업 — Colab 좌측 🔑 Secrets에 'UPSTAGE_API_KEY'를 등록한 뒤 실행합니다.
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
from openai import OpenAI
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro3"
print("Solar 준비 완료:", MODEL)

Solar 준비 완료: solar-pro3


## 1. OTel Tracer 설정
Trace(요청 전체) ⊃ Span(개별 작업) ⊃ Attribute(메타데이터)/Event(시점).
수집한 Span을 메모리에 모아 뒤에서 평가에 사용합니다(실서비스는 Phoenix/Arize/Langfuse로 전송합니다).

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

memory = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(memory))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("news-agent")
print("OTel Tracer 준비 완료 (in-memory exporter)")

OTel Tracer 준비 완료 (in-memory exporter)


## 2. Span으로 계측된 에이전트
에이전트의 각 단계를 `with tracer.start_as_current_span(...)`로 감싸고, **표준 attribute**를 기록합니다:
도구 span엔 `tool.name`·`tool.result_count`, LLM span엔 `llm.model`·`llm.input_tokens`·`llm.output_tokens`.

In [4]:
NEWS={"AI 반도체":["A사 AI 가속기 추론비용 40%↓","B사 온디바이스 AI 칩 양산"]}
def search_news(keyword):
    for k,v in NEWS.items():
        if any(w in keyword for w in k.split()) or k in keyword: return v
    return []

def traced_agent(question, keyword):
    with tracer.start_as_current_span("agent.run") as root:
        root.set_attribute("agent.input", question)
        # 1) 도구 호출 span
        with tracer.start_as_current_span("tool.search_news") as ts:
            hits = search_news(keyword)
            ts.set_attribute("tool.name","search_news")
            ts.set_attribute("tool.arg.keyword",keyword)
            ts.set_attribute("tool.result_count",len(hits))
        # 2) LLM 요약 span (실 Solar 호출 → 실제 토큰 기록)
        with tracer.start_as_current_span("llm.summarize") as ls:
            resp=client.chat.completions.create(model=MODEL,messages=[
                {"role":"system","content":"헤드라인을 한국어 한 문장으로 요약하라."},
                {"role":"user","content":str(hits)}])
            out=resp.choices[0].message.content.strip()
            ls.set_attribute("llm.model",MODEL)
            ls.set_attribute("llm.input_tokens",resp.usage.prompt_tokens)
            ls.set_attribute("llm.output_tokens",resp.usage.completion_tokens)
            root.set_attribute("agent.output",out)
            return out

answer=traced_agent("AI 반도체 뉴스 요약","AI 반도체")
print("답:",answer)

답: A사 AI 가속기 추론비용 40% 절감 및 B사 온디바이스 AI 칩 양산 시작.


## 3. 수집된 Trace 확인
exporter에 쌓인 Span을 들여다봅니다. 각 Span의 이름·attribute·소요시간이 보입니다 → **블랙박스가 열립니다.**

In [5]:
def dump_spans():
    for s in memory.get_finished_spans():
        dur_ms=(s.end_time-s.start_time)/1e6
        attrs={k:v for k,v in s.attributes.items()}
        print(f"[{s.name}]  {dur_ms:.1f}ms  {attrs}")
dump_spans()

[tool.search_news]  0.0ms  {'tool.name': 'search_news', 'tool.arg.keyword': 'AI 반도체', 'tool.result_count': 2}
[llm.summarize]  2172.6ms  {'llm.model': 'solar-pro3', 'llm.input_tokens': 104, 'llm.output_tokens': 23}
[agent.run]  2172.8ms  {'agent.input': 'AI 반도체 뉴스 요약', 'agent.output': 'A사 AI 가속기 추론비용 40% 절감 및 B사 온디바이스 AI 칩 양산 시작.'}


## 4. Span attribute 기반 Evaluator
Span의 attribute만으로 자동 평가합니다 — 어떤 프레임워크로 만든 에이전트든 **OTel만 내보내면** 동일하게 평가 가능.
예: 검색 결과가 0건이 아닌지, LLM 토큰이 예산을 넘지 않는지, 단계 지연이 임계 이내인지.

In [6]:
def evaluate_spans(spans, token_budget=2000, latency_budget_ms=15000):
    findings=[]
    for s in spans:
        a=s.attributes; dur_ms=(s.end_time-s.start_time)/1e6
        if a.get("tool.name")=="search_news" and a.get("tool.result_count",0)==0:
            findings.append(("검색 0건", s.name, "WARN"))
        if "llm.input_tokens" in a:
            tot=a["llm.input_tokens"]+a.get("llm.output_tokens",0)
            if tot>token_budget: findings.append((f"토큰 초과 {tot}>{token_budget}", s.name, "FAIL"))
        if dur_ms>latency_budget_ms:
            findings.append((f"지연 {dur_ms:.0f}ms>{latency_budget_ms}", s.name, "FAIL"))
    return findings

findings=evaluate_spans(memory.get_finished_spans())
print("평가 결과:", findings if findings else "✅ 모든 Span 기준 통과")

평가 결과: ✅ 모든 Span 기준 통과


## 5. (옵션) Phoenix로 시각화
실 서비스에선 콘솔 대신 **Arize Phoenix** 같은 UI로 Trace를 봅니다. Colab에서도 띄울 수 있습니다(선택).
```python
!pip install arize-phoenix opentelemetry-exporter-otlp
import phoenix as px
px.launch_app()   # Trace UI URL 출력 → OTLP exporter로 같은 span 전송
```

## 직접 해보기
토큰 예산을 빡빡하게 걸어 Evaluator가 위반을 잡아내는지 확인해 보세요. `evaluate_spans(..., token_budget=50)` 으로 호출해 FAIL이 나오는지 봅니다.

In [7]:
# TODO: evaluate_spans 를 token_budget=50 으로 호출해
#       LLM span이 '토큰 초과' FAIL로 잡히는지 확인하세요.


In [8]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# tight = evaluate_spans(memory.get_finished_spans(), token_budget=50)
# print("빡빡한 예산 평가:", tight)


## 6. 커스텀 Attribute — 평가에 필요한 신호를 직접 남기기
표준 attribute(토큰·지연)만으로는 '검색 품질' 같은 도메인 신호를 평가할 수 없습니다. `custom.*` attribute를 직접 계산해 Span에 기록합니다.

In [9]:
def traced_agent_custom(question, keyword):
    with tracer.start_as_current_span("agent.run.custom"):
        with tracer.start_as_current_span("tool.search_news") as ts:
            hits = search_news(keyword)
            ts.set_attribute("tool.result_count", len(hits))
            # 커스텀: 검색 결과가 질문 키워드를 포함하는 비율 = 관련성 점수
            rel = sum(any(w in h for w in keyword.split()) for h in hits) / len(hits) if hits else 0.0
            ts.set_attribute("custom.search_relevance", round(rel, 2))
        with tracer.start_as_current_span("llm.summarize") as ls:
            resp = client.chat.completions.create(model=MODEL, messages=[
                {"role":"system","content":"헤드라인을 한 문장으로 요약하라."},
                {"role":"user","content":str(hits)}])
            out = resp.choices[0].message.content.strip()
            ls.set_attribute("llm.output_tokens", resp.usage.completion_tokens)
            ls.set_attribute("custom.answer_length", len(out))   # 커스텀 메타데이터
        return out

print("답:", traced_agent_custom("AI 반도체 동향", "AI 반도체"))
print("→ custom.search_relevance / custom.answer_length 가 Span에 기록됩니다.")

답: A사 AI 가속기 추론비용 40%↓, B사 온디바이스 AI 칩 양산.
→ custom.search_relevance / custom.answer_length 가 Span에 기록됩니다.


## 7. 다단계 Span 계층 — 병목 찾기
실제 RAG는 검색→프롬프트구성→호출→포맷의 여러 단계입니다. 각 단계를 Span으로 감싸 **어느 단계가 가장 느린지** 찾습니다.

In [10]:
import time as _t

def rag_pipeline(keyword):
    with tracer.start_as_current_span("agent.rag"):
        with tracer.start_as_current_span("retriever.search") as s1:
            _t.sleep(0.05); hits = search_news(keyword); s1.set_attribute("stage", "search")
        with tracer.start_as_current_span("prompt.build") as s2:
            _t.sleep(0.02); prompt = "요약: " + str(hits); s2.set_attribute("stage", "prompt")
        with tracer.start_as_current_span("llm.call") as s3:
            resp = client.chat.completions.create(model=MODEL,
                messages=[{"role":"user","content":prompt}])
            out = resp.choices[0].message.content.strip(); s3.set_attribute("stage", "llm")
        with tracer.start_as_current_span("response.format") as s4:
            _t.sleep(0.01); out = "📰 " + out; s4.set_attribute("stage", "format")
        return out

rag_pipeline("전기차")
# Add force_flush to ensure spans are processed
provider.force_flush()
rag_spans = [s for s in memory.get_finished_spans() if s.attributes.get("stage")]
for s in rag_spans:
    print(f"  {s.name:18s} {(s.end_time - s.start_time) / 1e6:7.1f}ms")
slowest = max(rag_spans, key=lambda s: s.end_time - s.start_time)
print("→ 병목:", slowest.name)

  retriever.search      50.2ms
  prompt.build          20.2ms
  llm.call            2407.5ms
  response.format       10.2ms
→ 병목: llm.call


## 8. Event 기록 — 사건의 타임라인
Attribute가 '무엇'이라면 Event는 '언제 무슨 일이 일어났는지'입니다. 재시도·실패 같은 사건을 Span 안에 남깁니다.

In [11]:
def search_with_events(keyword):
    with tracer.start_as_current_span("tool.search.events") as ts:
        ts.add_event("search_started", {"keyword": keyword})
        hits = search_news(keyword)
        if not hits:
            ts.add_event("search_failed", {"reason": "no_results"})
        else:
            ts.add_event("search_succeeded", {"count": len(hits)})
        return hits

search_with_events("AI 반도체")
search_with_events("존재하지않는토픽")
# 실패 이벤트만 추출 → '언제 무엇이 실패했는지' 타임라인 진단
for s in memory.get_finished_spans():
    for ev in getattr(s, "events", []):
        if "fail" in ev.name:
            print(f"⚠️ {s.name}: {ev.name} {dict(ev.attributes)}")

⚠️ tool.search.events: search_failed {'reason': 'no_results'}


## 정리
- 에이전트를 **OTel Span**으로 계측하면 '왜 실패했는지'가 attribute로 남습니다(관측성).
- **Span attribute 기반 evaluator**는 프레임워크 독립적 — OTel만 내보내면 어디서든 평가할 수 있습니다.
- 토큰·지연 같은 운영 지표도 같은 Span에서 뽑습니다(이후 강의 Reliability와 연결됩니다).